# Overview

In this tutorial, we will go over how to carry out some basic molecular dynamics (MD) simulations using MLIPs. These MLIPs, in principle, can act as fast surrogates for density functional theory (DFT) and therefore can mimic an ab initio MD (AIMD) simulation at a very small fraction of the cost. Of course, in practice, they are ML models and therefore the agreement with AIMD is something that is not guaranteed.

Disclaimer: This is not an MD course, and there are _many_ important aspects one needs to know to run an MD simulation well enough to produce high-quality results. That said, hopefully this gives you a high level understanding of what's going on.

We will perform an MD simulation on the perovskite known as [methyl ammonium lead iodide](https://en.wikipedia.org/wiki/Methylammonium_lead_halide) (MAPbI3 for short), one of the first materials used in perovskite-based solar cells. The perovskite structure has [a cation in its void space](https://upload.wikimedia.org/wikipedia/commons/d/df/CH3NH3PbI3_structure.png), which we will watch jiggle around during the MD simulation.


# Documentation

The ASE documentation has a specific section for the [molecular dynamics methods](https://ase-lib.org/ase/md.html). There is also a brief tutorial in the [ASE 2023 Workshop](https://ase-workshop-2023.github.io/tutorial/10-dynamics/index.html).


# Setup


Note: If you are having issues installing or using Pymatgen, uninstall and reinstall matcalc. 


In [1]:
!uv pip install "ase>=3.28.0" "matcalc @ git+https://github.com/materialyzeai/matcalc.git" "upet>=0.2.2"

Using Python 3.13.11 environment at: /Users/christopherli/miniconda3/envs/cms
Resolved 90 packages in 748ms                                        
⠙ Preparing packages... (0/6)                                                   
⠙ Preparing packages... (0/6)-------------------     0 B/76.54 KiB           
⠙ Preparing packages... (0/6)------------------- 16.00 KiB/76.54 KiB         
⠙ Preparing packages... (0/6)m------------------ 32.00 KiB/76.54 KiB         
⠙ Preparing packages... (0/6)30m------------ 48.00 KiB/76.54 KiB         
⠙ Preparing packages... (0/6)30m------------ 48.00 KiB/76.54 KiB         
tqdm                 ------------------------------ 48.00 KiB/76.54 KiB
⠙ Preparing packages... (0/6)-------------------     0 B/2.68 MiB            
tqdm                 ------------------------------ 64.00 KiB/76.54 KiB
⠙ Preparing packages... (0/6)-------------------     0 B/2.68 MiB            
tqdm                 ------------------------------ 76.54 KiB/76.54 KiB
⠙ Preparing packa

In [2]:
!curl -L -o "MAPbI3.cif" "https://raw.githubusercontent.com/WMD-group/hybrid-perovskites/master/2014_cubic_halides_PBEsol/CH3NH3PbI3.cif"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1837  100  1837    0     0  37459      0 --:--:-- --:--:-- --:--:-- 38270


# Creating the Atoms Object


We will start by reading in the CIF we downloaded in the prior step.


In [3]:
from ase.io import read
from ase.visualize import view

atoms = read("MAPbI3.cif")
view(atoms, viewer="x3d")

The first thing we should do before setting up an MD simulation is to see how large our unit cell is. We do this because we want to make sure the simulation cell does not have ficticious self interactions across the periodic boundary conditions. Many MLIPs have a cutoff radius of 5 or 6 Å, but this depends on the model.


In [4]:
atoms.cell.lengths()

array([6.29012, 6.27389, 6.29704])

For good measure and for demonstration purposes, let's just double the lattice dimensions.


In [5]:
atoms *= (2, 2, 2)

# Defining the Calculator


Alright, now we must pick an ASE `Calculator` that defines the energies, forces, and stresses. For this demonstration, we will use one of the [UPET foundation models](https://github.com/lab-cosmo/upet), specifically the `upet-mad-xs` model because it is (as the name suggests) extra small and therefore extra fast (at the expense of lower accuracy).


In [6]:
from upet.calculator import UPETCalculator

calc = UPETCalculator(model="pet-mad-xs")

# Matcalc

We will start this demonstration by using `matcalc` to carry out the MD simulations. As a reminder, `matcalc` is just a wrapper around ASE and provides some convenient utility functions for various simulation tasks. It is not very flexible, so for advanced usage, it is better to use ASE directly, which we will do afterwards.


For this simulation, we are going to run an `NVT` simulation, meaning that the number of atoms (N), volume (V), and temperature (T) are held constant. In practice, the temperature is not held exactly constant because we use a "thermostat" that tries to maintain an average temperature, typically by rescaling particle velocities. That said, it's close enough. If you wanted to allow the unit cell volume to change, that would be an `NPT` simulation, which would require you to specify a constant pressure as well (as controlled by a "barostat"). For now, we will just carry out an NVT simulation for simplicity.

Hint: While the simulation is running, you can view the trajectory via the command line using `ase gui md.traj`! If you are using Colab, ase gui is not supported, so you will need to download md.traj locally on your computer to view it.


In [7]:
from matcalc import MDCalc
from ase.units import bar

MDCalc(
    calc,  # your ASE calculator
    ensemble="nvt",  # one of many possible MD ensembles and methods
    steps=500,  # number of steps to run the MD simulation
    timestep=1,  # timestep (in fs); 1 fs is typically a reasonable default
    temperature=300.0,  # user-defined temperature in K
    pressure=1 * bar,  # user-defined pressure (ignored in NVT!)
    trajfile="md.traj",  # write out ASE GUI-compatible trajectory file
    logfile="-",  # write out some metadata to a log file ("-" is printed to screen)
    relax_structure=True,  # relax the structure before the MD simulation
    fmax=0.05,  # eV/Å tolerance for the relaxation
    set_com_stationary=True,  # ensure the center of mass is not moving to start
).calc(atoms)

Time[ps]      Etot[eV]     Epot[eV]     Ekin[eV]    T[K]
0.0000        -431.2438    -434.7335       3.4897   281.2
0.0010        -431.2211    -434.4266       3.2055   258.3
0.0020        -431.1803    -433.8385       2.6582   214.2
0.0030        -431.1750    -433.5585       2.3835   192.1
0.0040        -431.2031    -433.6702       2.4671   198.8
0.0050        -431.2204    -433.7657       2.5453   205.1
0.0060        -431.2102    -433.5652       2.3551   189.8
0.0070        -431.1785    -433.2220       2.0435   164.7
0.0080        -431.1630    -433.2000       2.0369   164.2
0.0090        -431.1948    -433.6920       2.4973   201.2
0.0100        -431.2291    -434.2245       2.9953   241.4
0.0110        -431.2220    -434.2650       3.0430   245.2
0.0120        -431.1826    -433.8867       2.7041   217.9
0.0130        -431.1629    -433.6291       2.4662   198.7
0.0140        -431.1825    -433.7541       2.5716   207.2
0.0150        -431.2044    -433.9568       2.7525   221.8
0.0160        -

{'final_structure': Structure Summary
 Lattice
     abc : 12.58024 12.54778 12.594079999999998
  angles : 90.0014 90.36093 89.99929
  volume : 1987.987514846421
       A : np.float64(12.579990391234746) np.float64(0.00015395590678136197) np.float64(-0.07924765194627166)
       B : np.float64(0.0) np.float64(12.547779996254173) np.float64(-0.00030660010471128936)
       C : np.float64(0.0) np.float64(0.0) np.float64(12.594079999999998)
     pbc : True True True
 PeriodicSite: C (5.763, 6.324, 6.023) [0.4581, 0.504, 0.4812]
 PeriodicSite: N (0.819, 6.286, -0.08149) [0.06511, 0.501, -0.006049]
 PeriodicSite: H (5.741, 5.474, 5.311) [0.4564, 0.4363, 0.4246]
 PeriodicSite: H (5.746, 1.296, -1.465) [0.4568, 0.1033, -0.1134]
 PeriodicSite: H (5.583, 6.956, -1.368) [0.4438, 0.5544, -0.1058]
 PeriodicSite: H (0.2025, 1.656, 6.815) [0.0161, 0.132, 0.5412]
 PeriodicSite: H (1.566, 5.654, 6.557) [0.1245, 0.4506, 0.5214]
 PeriodicSite: H (0.9873, -0.817, 0.7207) [0.07848, -0.06511, 0.05772]
 Period

Our simulation above only ran for 500 fs (i.e. 0.5 ps). That is pretty short. In practice, you will typically want to simulate several nanoseconds at minimum, but you get the idea.


Now let's view the MD trajectory! You should be able to see the central ammonium cation jiggling around during the MD trajectory. This is something that would be difficult to observe from a standard structure relaxation at 0 K, as finite temperature imparts some thermal motion on the atoms and causes the atoms to move about their minimum energy configuration.

Note: In the ASE GUI, going to `View > Show Bonds` will make the simulation a bit easier to understand. This will draw bonds between atoms based on a geometry-based heuristic. There is no specific bonding information in the MLIP or in DFT itself.


In [8]:
trajectory = read("md.traj", index=":")
view(trajectory)

<Popen: returncode: None args: ['/Users/christopherli/miniconda3/envs/cms/bi...>

If you open `md.log`, you'll also see some information about the energy, temperature, and other properties as a function of the MD simulation length. Note that the temeprature will not be perfectly constant, as described above. There are optional keyword arguments in `MDCalc` that can be specified, which will control the time with which the thermostat rescales the velocities, but that is not important here.


# ASE

If you don't think you'll be doing MD simulations much, then you can stop here (feel free to change some of the parameters in the above `MDCalc` run though!).

In this part of the demo, we'll do the same analysis but using ASE directly, which gives you a bit more freedom.


First, let's redefine our `Atoms` object


In [9]:
atoms = read("MAPbI3.cif") * (2, 2, 2)
atoms.calc = calc

Now we will relax the structure before running the MD simulation.


In [10]:
from ase.optimize import BFGS

BFGS(atoms).run(fmax=0.05)

      Step     Time          Energy          fmax
BFGS:    0 14:37:04     -434.158325        0.541043
BFGS:    1 14:37:04     -434.296844        0.271626
BFGS:    2 14:37:04     -434.352203        0.264542
BFGS:    3 14:37:04     -434.379883        0.249975
BFGS:    4 14:37:04     -434.404022        0.228373
BFGS:    5 14:37:04     -434.441711        0.309354
BFGS:    6 14:37:04     -434.499298        0.439640
BFGS:    7 14:37:04     -434.541718        0.302063
BFGS:    8 14:37:04     -434.558777        0.104744
BFGS:    9 14:37:04     -434.565521        0.087900
BFGS:   10 14:37:05     -434.569427        0.088396
BFGS:   11 14:37:05     -434.576416        0.118581
BFGS:   12 14:37:05     -434.581696        0.083307
BFGS:   13 14:37:05     -434.586884        0.076433
BFGS:   14 14:37:05     -434.593933        0.119030
BFGS:   15 14:37:05     -434.606781        0.172999
BFGS:   16 14:37:05     -434.624603        0.231691
BFGS:   17 14:37:05     -434.639038        0.200644
BFGS:   18 14:

np.True_

With our updated `Atoms` object, it's time to run the MD simulation. The process is not terribly different from before, but we must do some things manually.


We start by giving the atoms a distribution of velocities to match the desired temperature. We then make sure the center of mass of the structure isn't mobile, otherwise it will just float across the simulation cell, which is annoying.


In [11]:
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.velocitydistribution import Stationary

MaxwellBoltzmannDistribution(atoms, temperature_K=300.0)
Stationary(atoms)

Now we need to pick an ensemble and method as outlined in the [ASE documentation](https://ase-lib.org/ase/md.html). Here, we will pack one of the [Constant NVT simulations](https://ase-lib.org/ase/md.html#constant-nvt-simulations-the-canonical-ensemble), of which ASE recommends several. One of the recommended ones is the Nosé-Hoover chain, which again is outside the scope of this course, but was the default when specifying `"nvt"` in `matcalc`. We will do that here. Note that the units are a bit funky.

For full details on the different keyword arguments, refer to the ASE documentation.


In [ ]:
from ase.units import fs
from ase.md.nose_hoover_chain import NoseHooverChainNVT

dyn = NoseHooverChainNVT(
    atoms,
    timestep=1.0 * fs,
    temperature_K=300.0,
    tdamp=100 * fs,
    trajectory="md2.traj",
    logfile="-",
)
dyn.run(steps=1000)

Time[ps]      Etot[eV]     Epot[eV]     Ekin[eV]    T[K]
0.0000        -428.5440    -431.8895       3.3456   269.6
0.0010        -428.5491    -431.9682       3.4191   275.5
0.0020        -428.5587    -432.0705       3.5118   283.0
0.0030        -428.5571    -432.0667       3.5097   282.8
0.0040        -428.5484    -431.9804       3.4320   276.6
0.0050        -428.5460    -431.9566       3.4106   274.8
0.0060        -428.5564    -432.0443       3.4879   281.1
0.0070        -428.5621    -432.1283       3.5662   287.4
0.0080        -428.5629    -432.1322       3.5693   287.6
0.0090        -428.5634    -432.0760       3.5127   283.1
0.0100        -428.5622    -432.0009       3.4387   277.1
0.0110        -428.5615    -431.9676       3.4061   274.5
0.0120        -428.5707    -432.0204       3.4497   278.0
0.0130        -428.5809    -432.0742       3.4934   281.5
0.0140        -428.5787    -432.0361       3.4575   278.6
0.0150        -428.5779    -431.9789       3.4011   274.1
0.0160        -

True

And we can once again view the trajectory.


In [15]:
trajectory = read("md2.traj", index=":")
view(trajectory)

<Popen: returncode: None args: ['/Users/christopherli/miniconda3/envs/cms/bi...>

This process is effectively the same as what was done when using `matcalc`, but it gives you more flexibility if needed. This time, I set it to run for 1 ps though.
